Combined horizontal stacked bar chart: NHDA (left) vs. RA (right) residential
building-type shares per Bavarian district (Landkreis / kreisfreie Stadt).

This script:

  0) Computes per-district NHDA and RA building-type shares directly from the
     GeoPackage produced by the earlier processing pipeline and writes them
     out as CSV files (so they exist as standalone intermediate products).
  1) Builds the NHDA table (the "master" row order/labels for the whole
     figure).
  2) Aligns the RA table to that exact same row order (districts missing RA
     data get an empty/zero bar instead of shifting the row order).
  3) Computes median building-type ratios per administrative region
     (Regierungsbezirk).
  4) Draws the combined figure: NHDA bars (left, mirrored, growing from the
     centre outward) | narrow middle column with the Regierungsbezirk name |
     RA bars (right, growing from the centre outward). District labels sit
     on the outer edge of each panel (left-aligned on the NHDA side,
     right-aligned on the RA side). Both panels stack building types in the
     same order, so MFH-AB touches the centre line on both sides.

In [ ]:
"""
Residential building-type shares per district (Landkreis / kreisfreie Stadt)
in Bavaria, computed separately for buildings inside NHDA areas and inside RA
areas.

This is derived directly from the original NHDA-only script. The expensive
steps -- loading the Bavaria-wide LoD2 buildings and assigning each building
to a district via a centroid spatial join -- are shared between NHDA and RA
and therefore only run ONCE. Only the "is this building inside area X?"
check and the per-district aggregation are done separately for NHDA and RA.

Output: two CSVs in the same wide format as the original NHDA script
(ARS, Name, lk_type, <BuildingType>_pct columns, ...), ready to be used by
the combined NHDA/RA plotting script.

IMPORTANT:
- The expensive building-to-district assignment is cached to disk (pickle).
  Set USE_CACHE = True and rerun to skip recomputation once it has run once.
"""

import math
import pickle
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

LOD2_NEW = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\LoD2\LoD2_2025_residential_types_thr_h11m.gpkg"

# Single GeoPackage containing BOTH the NHDA and the RA polygons, distinguished
# by the 'type' column ('NHDA' vs. 'RA').
AREAS_PATH = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg"
AREA_TYPE_COLUMN = 'type'
AREA_TYPES = {'NHDA': 'NHDA', 'RA': 'RA'}  # value in AREA_TYPE_COLUMN for each dataset

VG250_PATH = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
VG250_LAYER = "vg250_krs"

OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\building_nhda_ra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = OUTPUT_DIR / "building_district_assignment_cache.pkl"
USE_CACHE = True  # set True after the first successful run to skip the expensive step

MAKE_MAPS = True

COL_GEOMETRY = 'geometry'
COL_LK_KEY = 'LK_KEY'
COL_LK_NAME = 'LK_NAME'

BUILDING_TYPES = ['SFH-DB', 'SBD', 'TB', 'MFH-AB', 'unclassified res']

COLOR_MAP_TYPES = {
    'SFH-DB': "#82cbec",
    'SBD': "#febd2b",
    'TB': "#9aab4b",
    'MFH-AB': "#d94f21",
    'unclassified res': "#b3b3b3",
}

TARGET_CRS = 'EPSG:25832'
CRS_NOTE = 'CRS: EPSG:25832 - ETRS89 / UTM zone 32N'
GRID_STEP_M = 50000
MAP_PADDING_M = 10000

try:
    from matplotlib_map_utils import north_arrow, scale_bar
    HAS_MMU = True
except Exception:
    HAS_MMU = False

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 13,
    'axes.titlesize': 19,
    'axes.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
})


# ============================================================================
# MAP HELPERS (unchanged from the original NHDA script, reused for both maps)
# ============================================================================

def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(round(x / 1000))}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f'{int(round(y / 1000))}'))

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker='+', s=30, linewidths=1.0, color='#8a8a8a', alpha=0.95, zorder=4, clip_on=True)

    ax.tick_params(axis='both', which='major', labelsize=11, length=0, colors='#9B9999')


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor('#f1f1f1')
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)
    ax.set_xlabel('Easting (km) - UTM 32N', fontsize=12, color='#555555')
    ax.set_ylabel('Northing (km) - UTM 32N', fontsize=12, color='#555555')
    ax.text(0.01, 0.01, CRS_NOTE, transform=ax.transAxes, ha='left', va='bottom', fontsize=11, color='#555555')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color('#636262')
        ax.spines[side].set_linewidth(0.8)


def add_north_arrow(ax):
    if HAS_MMU:
        north_arrow(ax=ax, location='upper right', size='md', rotation={'degrees': 0},
                    aob={'bbox_to_anchor': (0.985, 0.985), 'bbox_transform': ax.transAxes,
                         'pad': 0.06, 'borderpad': 0.06, 'facecolor': 'none',
                         'edgecolor': 'none', 'alpha': 1.0, 'frameon': False})
        return
    ax.annotate('N', xy=(0.965, 0.97), xytext=(0.965, 0.86),
                xycoords='axes fraction', textcoords='axes fraction',
                ha='center', va='center', fontsize=16, fontweight='bold',
                arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0))


def add_scale_bar(ax):
    if HAS_MMU:
        scale_bar(ax=ax, location='lower right', size='xs', style='ticks',
                  bar={'projection': TARGET_CRS, 'unit': 'km', 'max': 50,
                       'major_div': 2, 'minor_div': 1, 'minor_type': 'none', 'reverse': False},
                  labels={'labels': ['0', '25', '50'], 'style': 'major', 'loc': 'below', 'fontsize': 9},
                  units={'loc': 'text', 'label': 'km'},
                  text={'fontfamily': 'sans-serif', 'fontsize': 9, 'textcolor': '#222222'},
                  aob={'pad': 0.0, 'borderpad': 1.0, 'facecolor': 'none',
                       'edgecolor': 'none', 'alpha': 1.0, 'frameon': False})
        return
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x, span_y = x1 - x0, y1 - y0
    bar_len, segment_len = 50_000, 25_000
    x_start = x1 - span_x * 0.25
    y_start = y0 + span_y * 0.032
    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color='#222222', linewidth=1.3, zorder=5)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color='#222222', linewidth=1.0, zorder=5)
    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, '0', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + segment_len, txt_y, '25', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + bar_len, txt_y, '50 km', ha='center', va='bottom', fontsize=9, color='#222222')


def add_legend_bottom_right(ax, handles, title):
    ax.legend(handles=handles, title=title, loc='lower left', bbox_to_anchor=(1.01, 0.02),
              borderaxespad=0.0, frameon=False, fontsize=13, title_fontsize=14,
              handlelength=2.0, labelspacing=0.6, borderpad=0.0)


def plot_dominant_type_map(gdf, output_dir, dataset_label):
    print(f"\nCreating map: Dominant building type inside {dataset_label} (district level)...")
    fig, ax = plt.subplots(figsize=(8.6, 9.4))
    fig.subplots_adjust(right=0.80, left=0.08, top=0.92, bottom=0.10)

    no_area_label = f'no {dataset_label}s'
    color_map_ext = {**COLOR_MAP_TYPES, no_area_label: '#e0e0e0'}
    colors = gdf['dominant_type'].map(color_map_ext).fillna('#e0e0e0')
    gdf.plot(ax=ax, color=colors, edgecolor='white', linewidth=0.6, zorder=2)

    present_types = [t for t in BUILDING_TYPES if t in gdf['dominant_type'].unique()]
    legend_elements = [mpatches.Patch(facecolor=COLOR_MAP_TYPES[t], edgecolor='white', label=t)
                        for t in present_types]
    if no_area_label in gdf['dominant_type'].unique():
        legend_elements.append(mpatches.Patch(facecolor='#e0e0e0', edgecolor='white', label=no_area_label))

    add_scientific_frame(ax, gdf)
    add_north_arrow(ax)
    add_scale_bar(ax)
    add_legend_bottom_right(ax, legend_elements, 'Dominant\nBuilding Type')

    output_dir.mkdir(parents=True, exist_ok=True)
    out_file = output_dir / f'dominant_type_{dataset_label.lower()}_bayern_landkreise.jpg'
    plt.savefig(out_file, bbox_inches='tight', facecolor='white', format='jpg')
    print(f"  Saved: {out_file}")
    plt.show()
    plt.close()


# ============================================================================
# SHARED (EXPENSIVE) STEP: load buildings + assign each one to a district
# ============================================================================

def load_bavaria_districts():
    print("Loading VG250 districts ...")
    lkr_raw = gpd.read_file(VG250_PATH, layer=VG250_LAYER)
    lkr_by = lkr_raw[lkr_raw["AGS"].astype(str).str.startswith("09")].to_crs(TARGET_CRS).copy()
    print(f"  {len(lkr_by):,} districts (Bavaria) | CRS: {lkr_by.crs}")

    type_candidates = [c for c in lkr_by.columns if "bez" in c.lower() or "art" in c.lower()]
    type_col = type_candidates[0] if type_candidates else None
    lk_type_series = lkr_by[type_col].astype(str) if type_col else lkr_by["GEN"].astype(str)
    lkr_by["lk_type"] = lk_type_series.apply(
        lambda x: "Landkreis" if ("landkreis" in x.lower() or "lkr" in x.lower()) else "Kreisfreie Stadt"
    )
    lkr = lkr_by[["AGS", "GEN", "lk_type", "geometry"]].rename(
        columns={"AGS": "lk_ags", "GEN": "lk_name"}
    ).copy()
    print(lkr["lk_type"].value_counts().to_string())
    return lkr


def build_buildings_with_district_assignment(lkr):
    """Load all residential LoD2 buildings (bbox-filtered to Bavaria) and
    assign each one to a district via a centroid spatial join. This is the
    expensive, dataset-independent step shared by NHDA and RA."""

    if USE_CACHE and CACHE_FILE.exists():
        print(f"\nLoading cached building-to-district assignment from {CACHE_FILE} ...")
        with open(CACHE_FILE, 'rb') as f:
            gdf_res, gdf_centroids = pickle.load(f)
        print(f"  {len(gdf_res):,} residential buildings (cached)")
        return gdf_res, gdf_centroids

    bavaria_bbox = tuple(lkr.total_bounds)
    print("\nLoading buildings (bbox-filtered, res_subclass only) ...")
    gdf_new = gpd.read_file(LOD2_NEW, bbox=bavaria_bbox, columns=["res_subclass"])
    if gdf_new.crs.to_epsg() != 25832:
        gdf_new = gdf_new.to_crs(TARGET_CRS)
    print(f"  {len(gdf_new):,} buildings | CRS: {gdf_new.crs}")

    gdf_res = gdf_new[gdf_new["res_subclass"].notna()].copy()
    del gdf_new
    print(f"  Residential with res_subclass: {len(gdf_res):,}")

    print("\nComputing building centroids ...")
    gdf_centroids = gdf_res[["geometry"]].copy()
    gdf_centroids["geometry"] = gdf_centroids.geometry.centroid

    print("Assigning buildings to districts ...")
    joined_lkr = gpd.sjoin(
        gdf_centroids,
        lkr[["lk_ags", "lk_name", "lk_type", "geometry"]],
        how="left",
        predicate="within",
    )
    joined_lkr = joined_lkr[~joined_lkr.index.duplicated(keep='first')]

    missing_mask = joined_lkr["lk_ags"].isna()
    if missing_mask.any():
        print(f"  Fallback nearest-district for {missing_mask.sum()} centroids ...")
        nearest = gpd.sjoin_nearest(
            gdf_centroids.loc[missing_mask],
            lkr[["lk_ags", "lk_name", "lk_type", "geometry"]],
            how="left",
        ).drop_duplicates(keep='first')
        joined_lkr.loc[missing_mask, "lk_ags"] = nearest["lk_ags"]
        joined_lkr.loc[missing_mask, "lk_name"] = nearest["lk_name"]
        joined_lkr.loc[missing_mask, "lk_type"] = nearest["lk_type"]

    gdf_res["lk_ags"] = joined_lkr["lk_ags"].reindex(gdf_res.index).values
    gdf_res["lk_name"] = joined_lkr["lk_name"].reindex(gdf_res.index).values
    gdf_res["lk_type"] = joined_lkr["lk_type"].reindex(gdf_res.index).values

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump((gdf_res, gdf_centroids), f)
    print(f"  Cached building-to-district assignment to {CACHE_FILE}")

    return gdf_res, gdf_centroids


# ============================================================================
# PER-DATASET STEP: flag buildings inside NHDA / RA, aggregate, export CSV
# ============================================================================

def compute_and_export(gdf_res, gdf_centroids, lkr, area_polygons, dataset_label):
    """Flag which buildings fall inside `area_polygons` (centroid-based),
    aggregate per-district building-type counts/percentages/dominant type,
    and export the result as a wide-format CSV matching the original NHDA
    script's output structure.
    """
    print(f"\n{'=' * 72}\n{dataset_label}: checking which buildings are inside the area polygons\n{'=' * 72}")

    area_flag = area_polygons[["geometry"]].copy()
    area_flag["_in_area"] = True

    in_area_join = gpd.sjoin(gdf_centroids, area_flag, how="left", predicate="within")
    in_area_flag = in_area_join["_in_area"].notna().groupby(level=0).any()

    gdf_res = gdf_res.copy()
    gdf_res["in_area"] = in_area_flag.reindex(gdf_res.index, fill_value=False).values
    print(f"  Buildings inside {dataset_label}: {gdf_res['in_area'].sum():,} / {len(gdf_res):,}")

    agg = (
        gdf_res.groupby(["lk_ags", "lk_name", "lk_type"])["in_area"]
        .agg(n_in_area=lambda x: x.sum(), n_outside_area=lambda x: (~x).sum(), n_total="count")
        .reset_index()
    )

    gdf_area_only = gdf_res[gdf_res["in_area"]].copy()
    print(f"  Buildings inside {dataset_label}: {len(gdf_area_only):,}")

    stats = gdf_area_only.groupby(["lk_ags", "res_subclass"]).size().reset_index(name='count')
    stats_pivot = stats.pivot_table(
        index="lk_ags", columns="res_subclass", values="count", fill_value=0
    ).reset_index()
    stats_pivot.columns.name = None
    for t in BUILDING_TYPES:
        if t not in stats_pivot.columns:
            stats_pivot[t] = 0

    type_cols = BUILDING_TYPES.copy()
    stats_pivot['total_buildings'] = stats_pivot[type_cols].sum(axis=1)
    for t in type_cols:
        stats_pivot[f'{t}_pct'] = (stats_pivot[t] / stats_pivot['total_buildings'] * 100).round(2)
    stats_pivot['dominant_type'] = stats_pivot[type_cols].idxmax(axis=1)
    stats_pivot['dominant_type_count'] = stats_pivot[type_cols].max(axis=1)
    stats_pivot['dominant_type_pct'] = (stats_pivot['dominant_type_count'] / stats_pivot['total_buildings'] * 100).round(2)

    print(f"  Dominant type computed for {len(stats_pivot)} districts")
    print(stats_pivot[['lk_ags', 'dominant_type', 'total_buildings']]
          .sort_values('total_buildings', ascending=False).head(10).to_string(index=False))

    lkr_plot = lkr.rename(columns={"lk_ags": COL_LK_KEY, "lk_name": COL_LK_NAME})
    stats_lk = stats_pivot.rename(columns={"lk_ags": COL_LK_KEY})

    gdf_result = lkr_plot.merge(stats_lk, on=COL_LK_KEY, how="left")
    no_area_label = f'no {dataset_label}s'
    gdf_result['dominant_type'] = gdf_result['dominant_type'].fillna(no_area_label)
    gdf_result['total_buildings'] = gdf_result['total_buildings'].fillna(0).astype(int)

    agg_renamed = agg.rename(columns={"lk_ags": COL_LK_KEY})
    gdf_result = gdf_result.merge(
        agg_renamed[[COL_LK_KEY, "n_in_area", "n_outside_area", "n_total"]],
        on=COL_LK_KEY, how="left"
    )
    print(f"\ngdf_result ({dataset_label}): {len(gdf_result)} districts")
    print("Dominant type distribution:")
    print(gdf_result['dominant_type'].value_counts())

    if MAKE_MAPS:
        plot_dominant_type_map(gdf_result, OUTPUT_DIR, dataset_label)

    stats_export_cols = (['total_buildings', 'dominant_type', 'dominant_type_count', 'dominant_type_pct']
                          + BUILDING_TYPES + [f'{t}_pct' for t in BUILDING_TYPES])
    available_cols = [c for c in stats_export_cols if c in gdf_result.columns]

    export_df = gdf_result[[COL_LK_KEY]].copy()
    export_df.insert(1, 'Name', gdf_result[COL_LK_NAME].values)
    export_df.insert(2, 'lk_type', gdf_result['lk_type'].values)
    for col in available_cols:
        export_df[col] = gdf_result[col].values
    for col in ["n_in_area", "n_outside_area", "n_total"]:
        if col in gdf_result.columns:
            export_df[col] = gdf_result[col].values

    export_df = export_df.rename(columns={COL_LK_KEY: 'ARS'})
    export_df['ARS'] = export_df['ARS'].astype(str).str.zfill(5)

    csv_file = OUTPUT_DIR / f'landkreis_{dataset_label.lower()}_buildingtype_stats.csv'
    export_df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"\nCSV saved: {csv_file}")

    return export_df, gdf_result


# ============================================================================
# MAIN PROGRAM
# ============================================================================

print("=" * 72)
print("RESIDENTIAL BUILDING TYPE SHARES - NHDA & RA - DISTRICT LEVEL")
print("=" * 72)

lkr = load_bavaria_districts()

print("\nLoading NHDA/RA area polygons ...")
areas = gpd.read_file(AREAS_PATH).to_crs(TARGET_CRS)
print(f"  {len(areas):,} area polygons total. Value counts for '{AREA_TYPE_COLUMN}':")
print(areas[AREA_TYPE_COLUMN].value_counts().to_string())

nhda_polygons = areas[areas[AREA_TYPE_COLUMN] == AREA_TYPES['NHDA']].copy()
ra_polygons = areas[areas[AREA_TYPE_COLUMN] == AREA_TYPES['RA']].copy()
print(f"  NHDA polygons: {len(nhda_polygons):,} | RA polygons: {len(ra_polygons):,}")

gdf_res, gdf_centroids = build_buildings_with_district_assignment(lkr)

nhda_export_df, nhda_gdf_result = compute_and_export(gdf_res, gdf_centroids, lkr, nhda_polygons, 'NHDA')
ra_export_df, ra_gdf_result = compute_and_export(gdf_res, gdf_centroids, lkr, ra_polygons, 'RA')

print(f"\nDone. Output directory: {OUTPUT_DIR}")

In [ ]:
"""
Combined horizontal stacked bar chart: NHDA (left) vs. RA (right) residential
building-type shares per Bavarian district (Landkreis / kreisfreie Stadt).

Reads the two CSVs produced by compute_nhda_ra_buildingtype_stats.py
(landkreis_nhda_buildingtype_stats.csv / landkreis_ra_buildingtype_stats.csv),
which already contain ARS, Name, lk_type and <BuildingType>_pct columns --
no GeoPackage/VG250 lookup needed here anymore.

Steps:
  1) Build the NHDA table (the "master" row order/labels for the whole
     figure).
  2) Align the RA table to that exact same row order (districts missing RA
     data get an empty/zero bar instead of shifting the row order).
  3) Compute median building-type ratios per administrative region
     (Regierungsbezirk).
  4) Draw the combined figure: NHDA bars (left, mirrored, growing from the
     centre outward) | narrow middle column with the Regierungsbezirk name |
     RA bars (right, growing from the centre outward). District labels sit
     on the outer edge of each panel (left-aligned on the NHDA side,
     right-aligned on the RA side). Both panels stack building types in the
     same order, so MFH-AB touches the centre line on both sides.
"""

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIG
# =============================================================================

INPUT_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\building_nhda_ra")
NHDA_CSV = INPUT_DIR / "landkreis_nhda_buildingtype_stats.csv"
RA_CSV = INPUT_DIR / "landkreis_ra_buildingtype_stats.csv"

OUTPUT_DIR = INPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

building_types = ['SFH-DB', 'SBD', 'TB', 'MFH-AB', 'unclassified res']

# Stacking order for the plot (from the centre line outward).
# MFH-AB touches the centre line on both panels.
plot_order = ['MFH-AB', 'TB', 'SBD', 'SFH-DB', 'unclassified res']

colors = {
    'SFH-DB': '#82cbec',
    'SBD': '#febd2b',
    'TB': '#9aab4b',
    'MFH-AB': '#d94f21',
    'unclassified res': '#b3b3b3',
}
regbez_map = {
    '091': 'Oberbayern', '092': 'Niederbayern', '093': 'Oberpfalz',
    '094': 'Oberfranken', '095': 'Mittelfranken', '096': 'Unterfranken', '097': 'Schwaben',
}
regbez_order = ['Oberbayern', 'Niederbayern', 'Oberpfalz', 'Oberfranken',
                 'Mittelfranken', 'Unterfranken', 'Schwaben']


def _clean_name(name):
    n = str(name).strip()
    for p in ['Landkreis ', 'Lkr. ', 'Kreisfreie Stadt ', 'Stadt ', 'Landeshauptstadt ']:
        if n.lower().startswith(p.lower()):
            return n[len(p):].strip()
    return n


def load_raw_shares(csv_path):
    """Load a per-district stats CSV and return an ARS-indexed table with
    raw shares (not normalised, not filtered, not sorted), plus a small
    district-metadata table (county name + lk_type)."""
    base_df = pd.read_csv(csv_path)

    if 'ARS' not in base_df.columns and 'LK_KEY' in base_df.columns:
        base_df = base_df.rename(columns={'LK_KEY': 'ARS'})
    if 'ARS' not in base_df.columns:
        raise ValueError(f'Missing ARS column in {csv_path}')

    if 'county' not in base_df.columns:
        for cand in ['Name', 'lk_name', 'LK_NAME']:
            if cand in base_df.columns:
                base_df = base_df.rename(columns={cand: 'county'})
                break
        else:
            raise ValueError(f'Missing district name column in {csv_path}')

    base_df['ARS'] = base_df['ARS'].astype(str).str.zfill(5)
    base_df['county'] = base_df['county'].astype(str).str.strip()

    if 'lk_type' in base_df.columns:
        district_meta = base_df[['ARS', 'county', 'lk_type']].drop_duplicates(subset=['ARS']).copy()
    else:
        district_meta = base_df[['ARS', 'county']].drop_duplicates(subset=['ARS']).copy()
        district_meta['lk_type'] = pd.NA

    pct_cols = [f'{bt}_pct' for bt in building_types if f'{bt}_pct' in base_df.columns]
    if pct_cols:
        wide = base_df[['ARS', 'county'] + pct_cols].drop_duplicates(subset=['ARS']).copy()
        wide = wide.rename(columns={f'{bt}_pct': bt for bt in building_types if f'{bt}_pct' in wide.columns})
    else:
        present = [bt for bt in building_types if bt in base_df.columns]
        if not present:
            raise ValueError(f'No building-type shares found in {csv_path}')
        wide = base_df[['ARS', 'county'] + present].drop_duplicates(subset=['ARS']).copy()

    for bt in building_types:
        if bt not in wide.columns:
            wide[bt] = 0.0
        wide[bt] = pd.to_numeric(wide[bt], errors='coerce').fillna(0.0)

    return wide.set_index('ARS'), district_meta


def _label(name, lk_type, name_is_ambiguous):
    n = _clean_name(name)
    t = '' if pd.isna(lk_type) else str(lk_type).strip().lower()
    if 'landkreis' in t and name_is_ambiguous:
        return f'{n} (Lkr.)'
    return n


# =============================================================================
# 1) NHDA: master row order / labels for the whole figure
# =============================================================================

wide_nhda_raw, district_meta_nhda = load_raw_shares(NHDA_CSV)
wide_nhda_raw = wide_nhda_raw.reset_index()

wide_nhda = wide_nhda_raw.copy()
wide_nhda['sum_share'] = wide_nhda[building_types].sum(axis=1)
has_nhda = wide_nhda['sum_share'] > 0
removed_no_nhda = int((~has_nhda).sum())
wide_nhda = wide_nhda.loc[has_nhda].copy()
if wide_nhda.empty:
    raise ValueError('After filtering, no districts with NHDA data remain to plot.')

wide_nhda[building_types] = wide_nhda[building_types].div(wide_nhda['sum_share'], axis=0) * 100
wide_nhda['sum_final'] = wide_nhda[building_types].sum(axis=1)
wide_nhda['SFH-DB'] = wide_nhda['SFH-DB'] + (100.0 - wide_nhda['sum_final'])

print(f'Excluded districts without NHDA data: {removed_no_nhda}')

wide_nhda['regbez_code'] = wide_nhda['ARS'].str[:3]
wide_nhda['regbez'] = wide_nhda['regbez_code'].map(regbez_map)
wide_nhda = wide_nhda.merge(district_meta_nhda[['ARS', 'lk_type']].drop_duplicates(subset=['ARS']), on='ARS', how='left')

wide_nhda['base_name'] = wide_nhda['county'].apply(_clean_name)
ambiguous_names = set(wide_nhda['base_name'].value_counts()[lambda s: s > 1].index)

wide_nhda['district_label'] = wide_nhda.apply(
    lambda r: _label(r['county'], r['lk_type'], r['base_name'] in ambiguous_names), axis=1
)
dup = wide_nhda['district_label'].duplicated(keep=False)
wide_nhda.loc[dup, 'district_label'] = wide_nhda.loc[dup, 'district_label'] + ' [' + wide_nhda.loc[dup, 'ARS'] + ']'

wide_nhda['regbez'] = pd.Categorical(wide_nhda['regbez'], categories=regbez_order, ordered=True)
wide_nhda = wide_nhda.sort_values(['regbez', 'district_label']).reset_index(drop=True)

# This is now the ONLY valid row order for the entire figure.
master_order = wide_nhda[['ARS', 'district_label', 'regbez']].copy()

# =============================================================================
# 2) RA: aligned to the NHDA master order (no independent filtering/sorting!)
# =============================================================================

wide_ra_raw, _ = load_raw_shares(RA_CSV)
wide_ra_raw = wide_ra_raw.reset_index()
wide_ra_raw['sum_share'] = wide_ra_raw[building_types].sum(axis=1)

mask = wide_ra_raw['sum_share'] > 0
wide_ra_raw.loc[mask, building_types] = wide_ra_raw.loc[mask, building_types].div(
    wide_ra_raw.loc[mask, 'sum_share'], axis=0
) * 100
wide_ra_raw.loc[mask, 'SFH-DB'] = wide_ra_raw.loc[mask, 'SFH-DB'] + (100.0 - wide_ra_raw.loc[mask, building_types].sum(axis=1))

wide_ra = master_order.merge(
    wide_ra_raw[['ARS'] + building_types], on='ARS', how='left'
)
missing_ra = wide_ra[building_types].isna().all(axis=1)
n_missing_ra = int(missing_ra.sum())
if n_missing_ra:
    print(f'Warning: {n_missing_ra} district(s) from the NHDA row order have no RA data -> empty bar.')
wide_ra[building_types] = wide_ra[building_types].fillna(0.0)

assert list(wide_ra['ARS']) == list(wide_nhda['ARS']), 'Row order between RA and NHDA has diverged!'

# =============================================================================
# 3) Median ratios per Regierungsbezirk
# =============================================================================

median_nhda = wide_nhda.groupby('regbez', observed=False)[building_types].median().reindex(regbez_order).dropna(how='all').round(2)
median_ra = wide_ra.groupby('regbez', observed=False)[building_types].median().reindex(regbez_order).dropna(how='all').round(2)

print('\nMedian NHDA building-type ratios (%) per Regierungsbezirk:')
display(median_nhda)
print('\nMedian RA building-type ratios (%) per Regierungsbezirk:')
display(median_ra)

median_nhda.to_csv(OUTPUT_DIR / 'median_building_type_ratios_nhda.csv', encoding='utf-8-sig')
median_ra.to_csv(OUTPUT_DIR / 'median_building_type_ratios_ra.csv', encoding='utf-8-sig')

# =============================================================================
# 4) Combined figure: NHDA (left) | Regierungsbezirk (middle) | RA (right)
# =============================================================================

line_spacing = 1.25
n = len(wide_nhda)
y = [i * line_spacing for i in range(n)]

fig, (ax_nhda, ax_mid, ax_ra) = plt.subplots(
    1, 3, figsize=(12.5, 11.69 * line_spacing), sharey=True,
    gridspec_kw={'width_ratios': [1.0, 0.16, 1.0], 'wspace': 0.0}
)

present_types = [t for t in plot_order if (wide_nhda[t].sum() > 0) or (wide_ra[t].sum() > 0)]

# --- Left: NHDA, mirrored (negative values, grows leftward from centre) ---
left = pd.Series(0.0, index=wide_nhda.index)
for t in present_types:
    vals = wide_nhda[t]
    ax_nhda.barh(y, -vals, left=-left, height=0.82, color=colors.get(t, '#7f7f7f'),
                 edgecolor='white', linewidth=0.4, label=t)
    left = left + vals

# --- Right: RA, regular (grows rightward from centre) ---
left = pd.Series(0.0, index=wide_ra.index)
for t in present_types:
    vals = wide_ra[t]
    ax_ra.barh(y, vals, left=left, height=0.82, color=colors.get(t, '#7f7f7f'),
               edgecolor='white', linewidth=0.4)
    left = left + vals

# NHDA axis (left, mirrored)
ax_nhda.set_xlim(-112, 0)
ax_nhda.set_xticks(range(-100, 1, 20))
ax_nhda.set_xticklabels([str(abs(v)) for v in range(-100, 1, 20)])
ax_nhda.set_xlabel('NHDA – share (%)', fontsize=8)
ax_nhda.tick_params(axis='x', labelsize=8.5)
ax_nhda.spines['top'].set_visible(False)
ax_nhda.spines['left'].set_visible(False)
ax_nhda.spines['right'].set_visible(False)
ax_nhda.grid(axis='x', linestyle='--', alpha=0.35, linewidth=0.6)
ax_nhda.set_axisbelow(True)
ax_nhda.set_yticks(y)
ax_nhda.set_yticklabels(wide_nhda['district_label'], fontsize=7.5)
ax_nhda.tick_params(axis='y', length=0, labelleft=True, labelright=False, pad=4)

# RA axis (right, regular)
ax_ra.set_xlim(0, 112)
ax_ra.set_xticks(range(0, 101, 20))
ax_ra.set_xlabel('RA – share (%)', fontsize=8)
ax_ra.tick_params(axis='x', labelsize=8.5)
ax_ra.spines['top'].set_visible(False)
ax_ra.spines['left'].set_visible(False)
ax_ra.spines['right'].set_visible(False)
ax_ra.grid(axis='x', linestyle='--', alpha=0.35, linewidth=0.6)
ax_ra.set_axisbelow(True)
ax_ra.yaxis.tick_right()
ax_ra.set_yticks(y)
ax_ra.set_yticklabels(wide_nhda['district_label'], fontsize=7.5)
ax_ra.tick_params(axis='y', length=0, labelright=True, labelleft=False, pad=4)

# Middle column: only the Regierungsbezirk name + divider lines.
ax_mid.set_xlim(-1, 1)
ax_mid.set_xticks([])
# IMPORTANT: since sharey=True, do NOT call ax_mid.set_yticks([]) here --
# that would clear the shared tick positions for ALL three axes. Just hide
# this axis' own tick marks/labels instead.
ax_mid.tick_params(axis='y', left=False, right=False, labelleft=False, labelright=False)
for spine in ax_mid.spines.values():
    spine.set_visible(False)

for ax in (ax_nhda, ax_mid, ax_ra):
    ax.invert_yaxis()
    ax.margins(y=0)
    ax.set_ylim((n - 0.5) * line_spacing, -0.5 * line_spacing)

# Regierungsbezirk divider lines (span all three panels) + centred name.
group_sizes = wide_nhda['regbez'].value_counts(sort=False).reindex(regbez_order).fillna(0).astype(int)
start = 0
for rb, size in group_sizes.items():
    if size == 0:
        continue
    end = start + size - 1
    center = (y[start] + y[end]) / 2
    ax_mid.text(0, center, rb, va='center', ha='center', rotation=90,
                fontsize=7.5, fontweight='bold')
    if end < n - 1:
        line_y = (y[end] + y[end + 1]) / 2
        for ax in (ax_nhda, ax_mid, ax_ra):
            ax.axhline(line_y, color='#444444', linewidth=1.6, alpha=0.9)
    start = end + 1

ax_nhda.set_title('NHDA', fontsize=11, fontweight='bold', loc='center')
ax_ra.set_title('RA', fontsize=11, fontweight='bold', loc='center')

ax_ra.legend(
    title='Building Type', ncol=1, loc='upper left', bbox_to_anchor=(1.02, 1.0),
    frameon=False, fontsize=8.5, title_fontsize=7, borderaxespad=0.0,
    handlelength=1.2, labelspacing=0.25, borderpad=0.0, columnspacing=0.6,
    handletextpad=0.4, markerscale=0.8, prop={'size': 8.5}
)

fig.subplots_adjust(left=0.14, right=0.80, top=0.96, bottom=0.045, wspace=0.0)

out_file = OUTPUT_DIR / 'nhda_ra_res_ratio_combined_horizontal.jpg'
plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {out_file}')

plt.show()